In [ ]:
!pip install llama-index llama-index-vector-stores-chroma llama-index-llms-huggingface-api llama-index-embeddings-huggingface -U -q
from huggingface_hub import login

login()

from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.core.agent.workflow import AgentWorkflow, ToolCallResult, AgentStream


def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two numbers"""
    return a - b


def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


def divide(a: int, b: int) -> int:
    """Divide two numbers"""
    return a / b


llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")

agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[subtract, multiply, divide, add],
    llm=llm,
    system_prompt="You are a math agent that can add, subtract, multiply, and divide numbers using provided tools.",
)

res = agent.run("the best beer in pairs")
print(res)

#### agent work flow 
- 利用内置tools 生成agent

In [ ]:
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.core.tools import FunctionTool

# define sample Tool -- type annotations, function names, and docstrings, are all included in parsed schemas!
def multiply(a: int, b: int) -> int:
    """Multiplies two integers and returns the resulting integer"""
    return a * b

# initialize llm
llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")

# function -> functionTool -> functionToolAgent 
agent = AgentWorkflow.from_tools_or_functions(
    [FunctionTool.from_defaults(multiply)],
    llm=llm
)

# 无状态调用
# agent.run("calcaulate 3 + 3");

# 通过Context 上下文对象实现有状态调用
from llama_index.core.workflow import Context

ctx = Context(agent)

# # 传入引用上下文
# response =  agent.run("My name is Bob",ctx = ctx)
# print(response)
# response  =  agent.run("What was my name again ?",ctx = ctx)
# print(response)


-  类似于 function -> functionTool -> functionToolAgent  , queryEngine -> queryEngineTool -> queryEngineAgent

In [ ]:
from llama_index.core.agent.workflow import (
    AgentWorkflow,
    FunctionAgent,
    ReActAgent,
)
from utils import get_query_engine,get_query_engine_tool

# Define some tools
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two numbers."""
    return a - b


# Create agent configs
# NOTE: we can use FunctionAgent or ReActAgent here.
# FunctionAgent works for LLMs with a function calling API.
# ReActAgent works for any LLM.
calculator_agent = ReActAgent(
    name="calculator",
    description="Performs basic arithmetic operations",
    system_prompt="You are a calculator assistant. Use your tools for any math operation.",
    tools=[add, subtract],
    llm=llm,
)

query_agent = ReActAgent(
    name="info_lookup",
    description="Looks up information about XYZ",
    system_prompt="Use your tool to query a RAG system to answer information about XYZ",
    tools=[get_query_engine_tool()],
    llm=llm
)

# 多agent 构成的agent 工作流
agent = AgentWorkflow(
    agents=[calculator_agent, query_agent], root_agent="calculator"
)

# Run the system
response = await agent.run(user_msg="Can you add 5 and 3?",ctx=ctx)

#### 定义工作流

In [ ]:
from llama_index.core.workflow import StartEvent,StopEvent,Workflow,step

# 继承 WorkFlow 实现
class MyWorkFlow(Workflow):
    @step
    async def my_step(self,ev: StartEvent) -> StopEvent:
        print("do my step")
        return StopEvent(result="Hello world step")
    

w = MyWorkFlow(timeout=10,verbose= False)
result =  w.run

<bound method Workflow.run of <__main__.MyWorkFlow object at 0x000001A1F9A863D0>>


In [ ]:
from llama_index.core.workflow import Event
import asyncio

# 继承事件类 
class ProcessingEvent(Event):
    intermediate_result: str

# 模拟一个多步骤agent ，通过事件驱动步骤循环： event->step->evet-step->end
class MultiStepWorkFlow(Workflow):
    @step
    async def step_one(self,ev: StartEvent) -> ProcessingEvent:
        return ProcessingEvent(intermediate_result="Step 1 complete")

    @step
    async def step_two(self,ev:ProcessingEvent) -> StopEvent:
        final_reslt = f"Finished processin:{ev.intermediate_result}"
        return StopEvent(result=final_reslt)

w = MultiStepWorkFlow(timeout=10,verbose=False)
result = await w.run()
result

'Finished processin:Step 1 complete'

#### create generic agent with loop and branchs

In [ ]:
from llama_index.core.workflow import Event
import random
from llama_index.utils.workflow import draw_all_possible_flows


class ProcessingEvent(Event):
    intermediate_result: str


class LoopEvent(Event):
    loop_output: str


class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent | LoopEvent) -> ProcessingEvent | LoopEvent:
        if random.randint(0, 1) == 0:
            print("Bad thing happened")
            return LoopEvent(loop_output="Back to step one.")
        else:
            print("Good thing happened")
            return ProcessingEvent(intermediate_result="First step complete.")

    @step
    async def step_two(self, ev: ProcessingEvent) -> StopEvent:
        # Use the intermediate result
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result=final_result)


w = MultiStepWorkflow(verbose=False)
result = await w.run()
result
# 绘制调用图
draw_all_possible_flows(w,"flow.html")

####  在工作流中添加上下文

In [ ]:
from llama_index.core.workflow import Context, StartEvent, StopEvent

# 方法定义中透传上下文
@step
async def query(self, ctx: Context, ev: StartEvent,msg:str) -> StopEvent:
    # store query in the context
    await ctx.store.set("query",msg)

    # do something with context and event
    val = f"get input {msg}"

    # retrieve query from the context
    query = await ctx.store.get("query")

    return StopEvent(result=val)